In [1]:
import pandas as pd
import numpy as np
from dataclasses import dataclass

In [2]:
df = pd.read_csv("Dump_data.csv", encoding='utf-8', encoding_errors='ignore')

# Standardize column names
df.columns = df.columns.str.strip()

df = df.replace(r'^\s*$', np.nan, regex=True)

df = df.apply(pd.to_numeric, errors='ignore')

In [3]:
df.columns

Index(['sample_location', 'PH', 'EC', 'TDS', 'TEMP', 'Ca', 'Mg', 'Na', 'K',
       'HCO3', 'SO4', 'Cl', 'NO3', 'CO3', 'Fe', 'Pb', 'Cu', 'Cr', 'Mn', 'Zn',
       'Ni', 'Co', 'Cd'],
      dtype='object')

In [4]:
WHO_LIMITS = {
"PH":8.5,
"T.D.S":500,
"TURBIDITY (NTU)":5,
"ALKALINITY (mg/l)":200,
"SO4- (mg/l)":250,
"NO3- (mg/l)":50,
"Cl- (mg/l)":250,
"HC03- (mg/l)":300,
"Ca2+  (mg/I)":75,
"Mg 2+ (mg/l)":50,
"K2+  (mg/I)":12,
" Na+  (mg/l)":200,
"FeT (mg/l)":0.3,
"Mn2+ (mg/I)":0.1,
"Cu 2+ (mg/I)":2,
"Zn 2+ (mg/I)":3
}

In [5]:
@dataclass
class WaterQualityIndex:

    df: pd.DataFrame

    def clean(self):

        data = self.df.copy()

        data = data.apply(pd.to_numeric, errors="coerce")

        return data

In [6]:
WaterQualityIndex(df)

WaterQualityIndex(df=   sample_location        PH           EC          TDS       TEMP         Ca  \
0              LP1  6.300000  1110.000000   530.000000  34.000000  55.180000   
1              LP2  7.300000   480.000000   240.000000  30.000000  56.000000   
2              LP3  6.400000   550.000000   280.000000  30.000000  56.850000   
3              LP4  6.600000   300.000000   150.000000  31.000000  64.000000   
4              LP5  7.200000   130.000000    60.000000  32.000000  58.010000   
5              LP6  6.200000   470.000000   230.000000  29.000000  63.200000   
6              LP7  6.100000   220.000000   110.000000  31.000000  55.600000   
7              AJ1  6.200000  3780.000000  1890.000000  27.000000  55.180000   
8              AJ2  6.600000   340.000000   170.000000  30.000000  54.400000   
9              AJ3  6.800000   370.000000   180.000000  31.000000  74.220000   
10             AJ4  5.800000   300.000000   150.000000  29.000000  63.870000   
11             AJ5 

In [7]:
class WQI(WaterQualityIndex):

    def calculate(self):

        data = self.clean()

        weights = {k:1/v for k,v in WHO_LIMITS.items()}

        Qi = pd.DataFrame()

        Wi = pd.Series(weights)

        for p in WHO_LIMITS:

            if p in data:

                Qi[p] = (data[p] / WHO_LIMITS[p]) * 100

        WQI = (Qi * Wi).sum(axis=1) / Wi.sum()

        return WQI

    def classify(self, values):

        return pd.cut(
            values,
            bins=[0,50,100,200,300,np.inf],
            labels=[
                "Excellent",
                "Good",
                "Poor",
                "Very Poor",
                "Unsuitable"
            ]
        )

In [8]:
build_index_table(df, WQI(df).calculate(), WQI(df).classify, lambda x: x, "WQI")

NameError: name 'build_index_table' is not defined

In [9]:
def run_index(index_class, df):

    model = index_class(df)

    values = model.calculate()

    table = pd.DataFrame()

    table["Index_Value"] = values

    if hasattr(model,"classify"):

        table["Class"] = model.classify(values)

    return table

In [10]:
run_index(WQI, df)

,Index_Value,Class
0,0.595434,Excellent
1,0.689947,Excellent
2,0.604885,Excellent
3,0.623788,Excellent
4,0.680496,Excellent
5,0.585983,Excellent
6,0.576531,Excellent
7,0.585983,Excellent
8,0.623788,Excellent
9,0.642691,Excellent


In [11]:
df.columns

Index(['sample_location', 'PH', 'EC', 'TDS', 'TEMP', 'Ca', 'Mg', 'Na', 'K',
       'HCO3', 'SO4', 'Cl', 'NO3', 'CO3', 'Fe', 'Pb', 'Cu', 'Cr', 'Mn', 'Zn',
       'Ni', 'Co', 'Cd'],
      dtype='object')

In [ ]:
class HPI(WaterQualityIndex):

    metals = [
    'Fe', 'Pb', 'Cu', 'Cr', 'Mn', 'Zn',
       'Ni', 'Co', 'Cd'
    ]

    limits = {
    "FeT (mg/l)":0.3,
    "Mn2+ (mg/I)":0.1,
    "Cu 2+ (mg/I)":2,
    "Zn 2+ (mg/I)":3
    }

    def calculate(self):

        data = self.clean()

        Qi = pd.DataFrame()

        Wi = {}

        for m in self.metals:

            if m in data:

                Si = self.limits[m]

                Wi[m] = 1/Si

                Qi[m] = (data[m] / Si) * 100

        Wi = pd.Series(Wi)

        HPI = (Qi * Wi).sum(axis=1) / Wi.sum()

        return HPI

    def classify(self, values):

        return pd.cut(
        values,
        bins=[0,50,100,200,np.inf],
        labels=[
        "Excellent",
        "Good",
        "Poor",
        "Very Poor"
        ])

In [15]:
run_index(HPI, df)

,Index_Value,Class
0,118.719608,Poor
1,127.286275,Poor
2,196.303922,Poor
3,117.737255,Poor
4,104.507843,Poor
5,383.668627,Very Poor
6,295.056863,Very Poor
7,259.856863,Very Poor
8,176.584314,Poor
9,99.729412,Good


In [16]:
class HEI(WaterQualityIndex):

    metals = {
    "FeT (mg/l)":0.3,
    "Mn2+ (mg/I)":0.1,
    "Cu 2+ (mg/I)":2,
    "Zn 2+ (mg/I)":3
    }

    def calculate(self):

        data = self.clean()

        ratios = []

        for m,limit in self.metals.items():

            if m in data:

                ratios.append(data[m]/limit)

        HEI = pd.concat(ratios,axis=1).sum(axis=1)

        return HEI

    def classify(self,values):

        return pd.cut(
        values,
        bins=[0,10,20,40,np.inf],
        labels=["Low","Moderate","High","Very High"])

In [17]:
run_index(HEI, df)

,Index_Value,Class
0,3.138333,Low
1,3.486667,Low
2,3.821667,Low
3,2.833333,Low
4,2.508333,Low
5,7.951667,Low
6,6.391667,Low
7,4.721667,Low
8,3.343333,Low
9,2.280000,Low


In [23]:
df.columns

Index(['S/nO', 'LOCATION', 'PH', 'T.D.S', 'CONDUCTIVITY (scm -1)', 'TEMP (0C)',
       'TURBIDITY (NTU)', 'ALKALINITY (mg/l)', 'SO4- (mg/l)', 'NO3- (mg/l)',
       'Cl- (mg/l)', 'HCO3 2- (mg/l)', 'Ca2+  (mg/I)', 'Mg 2+ (mg/l)',
       'K2+  (mg/I)', 'Na+  (mg/l)', 'FeT (mg/l)', 'Mn2+ (mg/I)',
       'Cu 2+ (mg/I)', 'Zn 2+ (mg/I)'],
      dtype='object')

In [92]:
import numpy as np

def calculate_sar(df):

    Na = df["Na__mgl"]
    Ca = df["Ca2__mgI"]
    Mg = df["Mg_2_mgl"]

    sar = Na / np.sqrt((Ca + Mg) / 2)

    return sar.rename("SAR")

In [104]:
import numpy as np

def classify_sar(sar_table):

    conditions = [
        sar_table["SAR"] < 10,
        (sar_table["SAR"] >= 10) & (sar_table["SAR"] < 18),
        (sar_table["SAR"] >= 18) & (sar_table["SAR"] < 26),
        sar_table["SAR"] >= 26
    ]

    classes = [
        "Excellent",
        "Good",
        "Doubtful",
        "Unsuitable"
    ]

    sar_table["SAR_Class"] = np.select(conditions, classes)

    return sar_table

In [98]:
def interpret_sar(sar_table):

    interpretation = {
        "Excellent": "Suitable for irrigation on all soil types",
        "Good": "Suitable with little sodium hazard",
        "Doubtful": "Use with soil management practices",
        "Unsuitable": "Not suitable for irrigation"
    }

    sar_table["SAR_Interpretation"] = sar_table["SAR_Class"].map(interpretation)

    return sar_table

In [107]:
import pandas as pd

def build_index_table(
        df,
        index_func,
        classify_func,
        interpret_func,
        index_name,
        export=True,
        file_format="csv"
    ):
    # Step 1 Calculate index
    index_values = index_func(df)

    # If dataframe returned, convert to series
    if isinstance(index_values, pd.DataFrame):
        index_values = index_values.iloc[:, 0]

    table = pd.DataFrame()
    table["Sample_No"] = df.index + 1
    table[index_name] = index_values.values

    # Step 2 Classification
    table = classify_func(table)

    # Step 3 Interpretation
    table = interpret_func(table)

    if export:
        if file_format == "csv":
            table.to_csv(f"{index_name}_results.csv", index=False)
        elif file_format == "excel":
            table.to_excel(f"{index_name}_results.xlsx", index=False)

    return table

In [108]:
sar_table = build_index_table(
    df,
    calculate_sar,
    classify_sar,
    interpret_sar,
    "SAR"
)

print(sar_table)

    Sample_No        SAR  SAR_Class                         SAR_Interpretation
0           1  14.120430       Good         Suitable with little sodium hazard
1           2   6.446962  Excellent  Suitable for irrigation on all soil types
2           3   7.732633  Excellent  Suitable for irrigation on all soil types
3           4   7.549667  Excellent  Suitable for irrigation on all soil types
4           5   3.401536  Excellent  Suitable for irrigation on all soil types
5           6   5.420447  Excellent  Suitable for irrigation on all soil types
6           7   6.016256  Excellent  Suitable for irrigation on all soil types
7           8   5.716572  Excellent  Suitable for irrigation on all soil types
8           9   3.864507  Excellent  Suitable for irrigation on all soil types
9          10   6.826296  Excellent  Suitable for irrigation on all soil types
10         11   7.981201  Excellent  Suitable for irrigation on all soil types
11         12   9.547882  Excellent  Suitable for ir

In [85]:
def interpret_sar(sar_table):

    interpretation = {
        "Excellent": "Suitable for irrigation on all soil types",
        "Good": "Suitable with little sodium hazard",
        "Doubtful": "Use with soil management practices",
        "Unsuitable": "Not suitable for irrigation"
    }

    sar_table["SAR_Interpretation"] = sar_table["SAR_Class"].map(interpretation)

    return sar_table

In [81]:
sar_calc = calculate_sar(df)
sar_class = classify_sar(sar_calc)
sar_result = interpret_sar(sar_class)
print("\nSAR Calculated Values")
print(sar_calc)
print("\nSAR Classified Values")
print(sar_class)
print("\nSAR Interpretation")
print(sar_result)


SAR Calculated Values
    Sample        SAR  SAR_Class                         SAR_Interpretation
0        0  14.120430       Good         Suitable with little sodium hazard
1        1   6.446962  Excellent  Suitable for irrigation on all soil types
2        2   7.732633  Excellent  Suitable for irrigation on all soil types
3        3   7.549667  Excellent  Suitable for irrigation on all soil types
4        4   3.401536  Excellent  Suitable for irrigation on all soil types
5        5   5.420447  Excellent  Suitable for irrigation on all soil types
6        6   6.016256  Excellent  Suitable for irrigation on all soil types
7        7   5.716572  Excellent  Suitable for irrigation on all soil types
8        8   3.864507  Excellent  Suitable for irrigation on all soil types
9        9   6.826296  Excellent  Suitable for irrigation on all soil types
10      10   7.981201  Excellent  Suitable for irrigation on all soil types
11      11   9.547882  Excellent  Suitable for irrigation on all 

In [134]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("+", "")
    .str.replace("-", "")
    .str.replace("(", "")
    .str.replace(")", "")
    .str.replace("/", "")
)

In [25]:
df.columns

Index(['SnO', 'LOCATION', 'PH', 'T.D.S', 'CONDUCTIVITY_scm_1', 'TEMP_0C',
       'TURBIDITY_NTU', 'ALKALINITY_mgl', 'SO4_mgl', 'NO3_mgl', 'Cl_mgl',
       'HCO3_2_mgl', 'Ca2__mgI', 'Mg_2_mgl', 'K2__mgI', 'Na__mgl', 'FeT_mgl',
       'Mn2_mgI', 'Cu_2_mgI', 'Zn_2_mgI'],
      dtype='object')

In [30]:
# Kelly Ratio calculation: 

def calculate_kr(df, na_col='Na__mgl', ca_col='Ca2__mgI', mg_col='Mg_2_mgl'):
    
    kr = df[na_col] / (df[ca_col] + df[mg_col])

    kr_table = pd.DataFrame({
        "Sample": df.index,
        "KR": kr
    })

    return kr_table

def classify_kr(kr_table):

    kr_table["KR_Class"] = np.where(
        kr_table["KR"] < 1,
        "Suitable",
        "Unsuitable"
    )

    return kr_table

def interpret_kr(kr_table):

    interpretation = {
        "Suitable": "Water is suitable for irrigation",
        "Unsuitable": "Water is unsuitable for irrigation due to excess sodium"
    }

    kr_table["KR_Interpretation"] = kr_table["KR_Class"].map(interpretation)

    return kr_table

In [31]:
kr_calc = calculate_kr(df)

kr_class = classify_kr(kr_calc)

kr_result = interpret_kr(kr_class)

print("\nKelly Ratio Calculated Values")
print(kr_calc)

print("\nKelly Ratio Classification")
print(kr_class[["Sample","KR_Class"]])

print("\nKelly Ratio Interpretation")
print(kr_result[["Sample","KR_Interpretation"]])


Kelly Ratio Calculated Values
    Sample        KR    KR_Class  \
0        0  1.091692  Unsuitable   
1        1  0.486234    Suitable   
2        2  0.690139    Suitable   
3        3  0.831594    Suitable   
4        4  0.184713    Suitable   
5        5  0.448569    Suitable   
6        6  0.414609    Suitable   
7        7  0.388945    Suitable   
8        8  0.282314    Suitable   
9        9  0.424625    Suitable   
10      10  0.340239    Suitable   
11      11  0.819508    Suitable   
12      12  0.646664    Suitable   
13      13  0.387373    Suitable   
14      14  0.295483    Suitable   
15      15  0.577694    Suitable   
16      16  0.542302    Suitable   
17      17  0.549987    Suitable   
18      18  0.385377    Suitable   
19      19  0.713140    Suitable   
20      20  0.452853    Suitable   

                                    KR_Interpretation  
0   Water is unsuitable for irrigation due to exce...  
1                    Water is suitable for irrigation  
2       

In [111]:
# Merge all KR outputs into one table
kr_export = pd.DataFrame()

kr_export["Sample"] = kr_calc["Sample"]
kr_export["KR"] = kr_calc["KR"]
kr_export["KR_Class"] = kr_class["KR_Class"]
kr_export["KR_Interpretation"] = kr_result["KR_Interpretation"]

# Export to CSV
kr_export.to_csv("Kelly_Ratio_Results.csv", index=False)

print("Kelly Ratio results exported successfully.")

Kelly Ratio results exported successfully.


In [35]:
df.columns

Index(['SnO', 'LOCATION', 'PH', 'T.D.S', 'CONDUCTIVITY_scm_1', 'TEMP_0C',
       'TURBIDITY_NTU', 'ALKALINITY_mgl', 'SO4_mgl', 'NO3_mgl', 'Cl_mgl',
       'HCO3_2_mgl', 'Ca2__mgI', 'Mg_2_mgl', 'K2__mgI', 'Na__mgl', 'FeT_mgl',
       'Mn2_mgI', 'Cu_2_mgI', 'Zn_2_mgI'],
      dtype='object')

In [36]:
# Magnesium Hazard Calculation:

def calculate_mh(df, ca_col='Ca2__mgI', mg_col='Mg_2_mgl'):

    mh = (df[mg_col] / (df[ca_col] + df[mg_col])) * 100

    mh_table = pd.DataFrame({
        "Sample": df.index,
        "MH (%)": mh
    })

    return mh_table

def classify_mh(mh_table):

    mh_table["MH_Class"] = np.where(
        mh_table["MH (%)"] < 50,
        "Suitable",
        "Unsuitable"
    )

    return mh_table

def interpret_mh(mh_table):

    interpretation = {
        "Suitable": "Water is suitable for irrigation; magnesium hazard is low",
        "Unsuitable": "Excess magnesium may reduce soil quality and crop yield"
    }

    mh_table["MH_Interpretation"] = mh_table["MH_Class"].map(interpretation)

    return mh_table


mh_calc = calculate_mh(df)

mh_class = classify_mh(mh_calc)

mh_result = interpret_mh(mh_class)

print("\nMagnesium Hazard Calculated Values")
print(mh_calc)

print("\nMagnesium Hazard Classification")
print(mh_class[["Sample","MH_Class"]])

print("\nMagnesium Hazard Interpretation")
print(mh_result[["Sample","MH_Interpretation"]])





Magnesium Hazard Calculated Values
    Sample     MH (%)  MH_Class  \
0        0  34.118350  Suitable   
1        1  21.137656  Suitable   
2        2  30.301099  Suitable   
3        3  14.486775  Suitable   
4        4  11.229063  Suitable   
5        5  11.970963  Suitable   
6        6   7.275836  Suitable   
7        7  12.637719  Suitable   
8        8   9.435372  Suitable   
9        9  32.138988  Suitable   
10      10  20.561189  Suitable   
11      11  13.923678  Suitable   
12      12  13.091506  Suitable   
13      13  21.337926  Suitable   
14      14   5.837087  Suitable   
15      15  15.946381  Suitable   
16      16  18.580915  Suitable   
17      17  12.772838  Suitable   
18      18  15.990424  Suitable   
19      19  28.747687  Suitable   
20      20  17.285677  Suitable   

                                    MH_Interpretation  
0   Water is suitable for irrigation; magnesium ha...  
1   Water is suitable for irrigation; magnesium ha...  
2   Water is suitable for

In [114]:
mh_table = build_index_table(
    df,
    calculate_mh,
    classify_mh,
    interpret_mh,
    "MH (%)"
)

print(mh_table)

    Sample_No  MH (%)  MH_Class  \
0           1       0  Suitable   
1           2       1  Suitable   
2           3       2  Suitable   
3           4       3  Suitable   
4           5       4  Suitable   
5           6       5  Suitable   
6           7       6  Suitable   
7           8       7  Suitable   
8           9       8  Suitable   
9          10       9  Suitable   
10         11      10  Suitable   
11         12      11  Suitable   
12         13      12  Suitable   
13         14      13  Suitable   
14         15      14  Suitable   
15         16      15  Suitable   
16         17      16  Suitable   
17         18      17  Suitable   
18         19      18  Suitable   
19         20      19  Suitable   
20         21      20  Suitable   

                                    MH_Interpretation  
0   Water is suitable for irrigation; magnesium ha...  
1   Water is suitable for irrigation; magnesium ha...  
2   Water is suitable for irrigation; magnesium ha...  
3   W

In [115]:
# Merge all mh outputs into one table
mh_export = pd.DataFrame()

mh_export["Sample"] = mh_calc["Sample"]
mh_export["MH"] = mh_calc["MH (%)"]
mh_export["MH_Class"] = mh_class["MH_Class"]
mh_export["MH_Interpretation"] = mh_result["MH_Interpretation"]

# Export to CSV
mh_export.to_csv("Magnesium_Hazard_Results.csv", index=False)

print("Magnesium Hazard results exported successfully.")

Magnesium Hazard results exported successfully.


In [ ]:
run_index(KellyRatio, df)

,Index_Value
0,1.091692
1,0.486234
2,0.690139
3,0.831594
4,0.184713
5,0.448569
6,0.414609
7,0.388945
8,0.282314
9,0.424625


In [37]:
df.columns

Index(['SnO', 'LOCATION', 'PH', 'T.D.S', 'CONDUCTIVITY_scm_1', 'TEMP_0C',
       'TURBIDITY_NTU', 'ALKALINITY_mgl', 'SO4_mgl', 'NO3_mgl', 'Cl_mgl',
       'HCO3_2_mgl', 'Ca2__mgI', 'Mg_2_mgl', 'K2__mgI', 'Na__mgl', 'FeT_mgl',
       'Mn2_mgI', 'Cu_2_mgI', 'Zn_2_mgI'],
      dtype='object')

In [38]:
# Permeability Index
def calculate_pi(df, na_col='Na__mgl', ca_col='Ca2__mgI', mg_col='Mg_2_mgl', hco3_col='HCO3_2_mgl'):

    pi = ((df[na_col] + np.sqrt(df[hco3_col])) /
          (df[ca_col] + df[mg_col] + df[na_col])) * 100

    pi_table = pd.DataFrame({
        "Sample": df.index,
        "PI (%)": pi
    })

    return pi_table

def classify_pi(pi_table):

    conditions = [
        pi_table["PI (%)"] > 75,
        (pi_table["PI (%)"] >= 25) & (pi_table["PI (%)"] <= 75),
        pi_table["PI (%)"] < 25
    ]

    classes = [
        "Class I",
        "Class II",
        "Class III"
    ]

    pi_table["PI_Class"] = np.select(conditions, classes)

    return pi_table

def interpret_pi(pi_table):

    interpretation = {
        "Class I": "Excellent irrigation water; no permeability problem",
        "Class II": "Moderately suitable irrigation water",
        "Class III": "Unsuitable water; soil permeability likely affected"
    }

    pi_table["PI_Interpretation"] = pi_table["PI_Class"].map(interpretation)

    return pi_table

In [118]:
pi_calc = calculate_pi(df)

pi_class = classify_pi(pi_calc)

pi_result = interpret_pi(pi_class)

print("\nPermeability Index Calculated Values")
print(pi_calc)

print("\nPermeability Index Classification")
print(pi_class[["Sample","PI_Class"]])

print("\nPermeability Index Interpretation")
print(pi_result[["Sample","PI_Interpretation"]])


Permeability Index Calculated Values
    Sample     PI (%)   PI_Class  \
0        0  59.599635   Class II   
1        1  42.637386   Class II   
2        2  47.085729   Class II   
3        3  64.325460   Class II   
4        4  23.041930  Class III   
5        5  46.674780   Class II   
6        6  38.515618   Class II   
7        7  37.429766   Class II   
8        8  34.249158   Class II   
9        9  35.345776   Class II   
10      10        NaN          0   
11      11  57.374319   Class II   
12      12  53.197617   Class II   
13      13  35.707017   Class II   
14      14  27.637997   Class II   
15      15  48.851278   Class II   
16      16  46.988038   Class II   
17      17  45.005608   Class II   
18      18  33.932935   Class II   
19      19  62.556231   Class II   
20      20  40.252432   Class II   

                                    PI_Interpretation  
0                Moderately suitable irrigation water  
1                Moderately suitable irrigation water  
2

In [116]:
# Merge all permeability_index outputs into one table
pi_export = pd.DataFrame()

pi_export["Sample"] = pi_calc["Sample"]
pi_export["MH"] = pi_calc["PI (%)"]
pi_export["MH_Class"] = pi_class["PI_Class"]
pi_export["MH_Interpretation"] = pi_result["PI_Interpretation"]

# Export to CSV
pi_export.to_csv("Permeability_Index_Results.csv", index=False)

print("Permeability Index results exported successfully.")

Permeability Index results exported successfully.


In [46]:

def calculate_sodium(df, na_col='Na__mgl', ca_col='Ca2__mgI', mg_col='Mg_2_mgl', k_col='K2__mgI'):
        
        Sodium = (df[na_col] + df[k_col]) * 100 / (df[ca_col] + df[mg_col] + df[na_col] + df[k_col])

        sodiumperc = pd.DataFrame({
            "Sample": df.index,
            "Sodium (%)": Sodium
        })

        return sodiumperc
def classify_sodium(sodiumperc):
        return np.where(sodiumperc["Sodium (%)"] < 50, "Suitable", "Unsuitable")
    
def interpret_sodium(sodiumperc):
        interpretation = {
            "Suitable": "Water is suitable for irrigation; sodium percentage is low",
            "Unsuitable": "High sodium percentage may lead to soil degradation and reduced crop yield"
        }
        sodiumperc["Sodium_Interpretation"] = sodiumperc["Sodium (%)"].map(interpretation)
        return sodiumperc

In [50]:
sodium_calc = calculate_sodium(df)

sodium_calc["Sodium_Class"] = classify_sodium(sodium_calc)

sodium_result = interpret_sodium(sodium_calc)

print("\nSodium Percentage Calculated Values")
print(sodium_calc)

print("\nSodium Percentage Classification")
print(sodium_result[["Sample","Sodium_Class"]])

print("\nSodium Percentage Interpretation")
print(sodium_result[["Sample","Sodium_Class","Sodium_Interpretation"]])


Sodium Percentage Calculated Values
    Sample  Sodium (%) Sodium_Class Sodium_Interpretation
0        0   56.925850   Unsuitable                   NaN
1        1   51.452557   Unsuitable                   NaN
2        2   41.440433     Suitable                   NaN
3        3   48.124371     Suitable                   NaN
4        4   34.380805     Suitable                   NaN
5        5   33.385036     Suitable                   NaN
6        6   31.279373     Suitable                   NaN
7        7   34.083974     Suitable                   NaN
8        8   26.373281     Suitable                   NaN
9        9   32.455177     Suitable                   NaN
10      10   51.286319   Unsuitable                   NaN
11      11   46.776976     Suitable                   NaN
12      12   40.753516     Suitable                   NaN
13      13   46.865744     Suitable                   NaN
14      14   28.811295     Suitable                   NaN
15      15   38.868494     Suitable

In [127]:
import pandas as pd

# Calculate sodium percentage
sodium_calc = calculate_sodium(df)

# Classify sodium percentage
sodium_calc["Sodium_Class"] = classify_sodium(sodium_calc)

# Interpret the classification
sodium_result = interpret_sodium(sodium_calc)

# Prepare export table
sodium_export = pd.DataFrame()
sodium_export["Sample"] = sodium_result["Sample"]
sodium_export["Sodium_percentage"] = sodium_result["Sodium (%)"]
sodium_export["Sodium_Class"] = sodium_result["Sodium_Class"]
sodium_export["Sodium_Interpretation"] = sodium_result["Sodium_Interpretation"]

# Export to CSV
sodium_export.to_csv("Sodium_Percentage_Results.csv", index=False)

print("✅ Sodium Percentage results exported successfully.")
print(f"   Total samples: {len(sodium_export)}")

✅ Sodium Percentage results exported successfully.
   Total samples: 21


In [51]:
df.columns


Index(['SnO', 'LOCATION', 'PH', 'T.D.S', 'CONDUCTIVITY_scm_1', 'TEMP_0C',
       'TURBIDITY_NTU', 'ALKALINITY_mgl', 'SO4_mgl', 'NO3_mgl', 'Cl_mgl',
       'HCO3_2_mgl', 'Ca2__mgI', 'Mg_2_mgl', 'K2__mgI', 'Na__mgl', 'FeT_mgl',
       'Mn2_mgI', 'Cu_2_mgI', 'Zn_2_mgI'],
      dtype='object')

In [52]:
class HPI(WaterQualityIndex):

    metals = [
    "FeT_mg/l",
    "Mn2_mg/I",
    "Cu_2_mg/I",
    "Zn_2_mg/I"
    ]

    limits = {
    "FeT_mg/l":0.3,
    "Mn2_mg/I":0.1,
    "Cu_2_mg/I":2,
    "Zn_2_mg/I":3
    }

    def calculate(self):

        data = self.clean()

        Qi = pd.DataFrame()

        Wi = {}

        for m in self.metals:

            if m in data:

                Si = self.limits[m]

                Wi[m] = 1/Si

                Qi[m] = (data[m] / Si) * 100

        Wi = pd.Series(Wi)

        HPI = (Qi * Wi).sum(axis=1) / Wi.sum()

        return HPI

    def classify(self, values):

        return pd.cut(
        values,
        bins=[0,50,100,200,np.inf],
        labels=[
        "Excellent",
        "Good",
        "Poor",
        "Very Poor"
        ])

In [53]:
run_index(HPI, df)

,Index_Value,Class


In [57]:
df.columns  

Index(['SnO', 'LOCATION', 'PH', 'T.D.S', 'CONDUCTIVITY_scm_1', 'TEMP_0C',
       'TURBIDITY_NTU', 'ALKALINITY_mgl', 'SO4_mgl', 'NO3_mgl', 'Cl_mgl',
       'HCO3_2_mgl', 'Ca2__mgI', 'Mg_2_mgl', 'K2__mgI', 'Na__mgl', 'FeT_mgl',
       'Mn2_mgI', 'Cu_2_mgI', 'Zn_2_mgI'],
      dtype='object')

In [58]:
metals = [
    "FeT_mgl",
    "Mn2_mgI",
    "Cu_2_mgI",
    "Zn_2_mgI"
    ]

limits = {
    "FeT_mgl":0.3,
    "Mn2_mgI":0.1,
    "Cu_2_mgI":2,
    "Zn_2_mgI":3
    }

In [61]:
#Heavy metal pollution index (HPI) is a method used to assess the overall quality of water with respect to heavy metal contamination. It provides a single value that represents the combined effect of multiple heavy metals present in the water. The HPI is calculated using the concentrations of various heavy metals and their respective permissible limits set by regulatory agencies.

def calculate_hpi(df, limits):

    weights = {metal: 1/limit for metal, limit in limits.items()}

    qi = {}
    wi_qi = {}

    for metal in limits:
        qi[metal] = (df[metal] / limits[metal]) * 100
        wi_qi[metal] = qi[metal] * weights[metal]

    qi_df = pd.DataFrame(qi)
    wi_qi_df = pd.DataFrame(wi_qi)

    hpi = wi_qi_df.sum(axis=1) / sum(weights.values())

    hpi_table = pd.DataFrame({
        "Sample": df.index,
        "HPI": hpi
    })

    return hpi_table

def classify_hpi(hpi_table):

    hpi_table["HPI_Class"] = np.where(
        hpi_table["HPI"] < 100,
        "Low Pollution",
        "Critical Pollution"
    )

    return hpi_table

def interpret_hpi(hpi_table):

    interpretation = {
        "Low Pollution": "Water is safe with respect to heavy metals",
        "Critical Pollution": "Water is heavily contaminated with heavy metals"
    }

    hpi_table["HPI_Interpretation"] = hpi_table["HPI_Class"].map(interpretation)

    return hpi_table

In [62]:
hpi_calc = calculate_hpi(df, limits)
hpi_class = classify_hpi(hpi_calc)
hpi_result = interpret_hpi(hpi_class)

print("\nHPI Calculated Values")
print(hpi_calc)

print("\nHPI Classification")
print(hpi_class[["Sample","HPI_Class"]])

print("\nHPI Interpretation")
print(hpi_result[["Sample","HPI_Interpretation"]])


HPI Calculated Values
    Sample          HPI           HPI_Class  \
0        0   118.719608  Critical Pollution   
1        1   127.286275  Critical Pollution   
2        2   196.303922  Critical Pollution   
3        3   117.737255  Critical Pollution   
4        4   104.507843  Critical Pollution   
5        5   383.668627  Critical Pollution   
6        6   295.056863  Critical Pollution   
7        7   259.856863  Critical Pollution   
8        8   176.584314  Critical Pollution   
9        9    99.729412       Low Pollution   
10      10  1118.670588  Critical Pollution   
11      11    52.672549       Low Pollution   
12      12   113.119608  Critical Pollution   
13      13   222.066667  Critical Pollution   
14      14    95.792157       Low Pollution   
15      15   369.723529  Critical Pollution   
16      16   491.927451  Critical Pollution   
17      17  2669.262745  Critical Pollution   
18      18   317.049020  Critical Pollution   
19      19   151.015686  Critical Pol

In [129]:
import pandas as pd

# Calculate HPI
hpi_calc = calculate_hpi(df, limits)

# Classify HPI values
hpi_class = classify_hpi(hpi_calc)

# Interpret HPI classification
hpi_result = interpret_hpi(hpi_class)

# Display results (as in your original code)
print("\nHPI Calculated Values")
print(hpi_calc)

print("\nHPI Classification")
print(hpi_class[["Sample","HPI_Class"]])

print("\nHPI Interpretation")
print(hpi_result[["Sample","HPI_Interpretation"]])

# === EXPORT TO CSV ===

# Method 1: If all functions return DataFrames with same structure as sodium
hpi_export = pd.DataFrame()

# Check which DataFrame has all the columns we need
# Usually the final result (hpi_result) contains everything
hpi_export["Sample"] = hpi_result["Sample"]
hpi_export["HPI_Value"] = hpi_result["HPI"]  # Adjust column name as needed
hpi_export["HPI_Class"] = hpi_result["HPI_Class"]
hpi_export["HPI_Interpretation"] = hpi_result["HPI_Interpretation"]

# Export to CSV
hpi_export.to_csv("HPI_Results.csv", index=False)

print("\n✅ HPI Results exported successfully to 'HPI_Results.csv'")
print(f"   Total samples: {len(hpi_export)}")


HPI Calculated Values
    Sample          HPI           HPI_Class  \
0        0   118.719608  Critical Pollution   
1        1   127.286275  Critical Pollution   
2        2   196.303922  Critical Pollution   
3        3   117.737255  Critical Pollution   
4        4   104.507843  Critical Pollution   
5        5   383.668627  Critical Pollution   
6        6   295.056863  Critical Pollution   
7        7   259.856863  Critical Pollution   
8        8   176.584314  Critical Pollution   
9        9    99.729412       Low Pollution   
10      10  1118.670588  Critical Pollution   
11      11    52.672549       Low Pollution   
12      12   113.119608  Critical Pollution   
13      13   222.066667  Critical Pollution   
14      14    95.792157       Low Pollution   
15      15   369.723529  Critical Pollution   
16      16   491.927451  Critical Pollution   
17      17  2669.262745  Critical Pollution   
18      18   317.049020  Critical Pollution   
19      19   151.015686  Critical Pol

In [64]:
# Heavy metal Evaluation Index (HEI) is a method used to assess the overall quality of water with respect to heavy metal contamination. It provides a single value that represents the combined effect of multiple heavy metals present in the water. The HEI is calculated using the concentrations of various heavy metals and their respective permissible limits set by regulatory agencies.

def calculate_hei(df, limits):

    hei = pd.Series(0, index=df.index)

    for metal in limits:
        hei += df[metal] / limits[metal]

    hei_table = pd.DataFrame({
        "Sample": df.index,
        "HEI": hei
    })

    return hei_table

def classify_hei(hei_table):

    conditions = [
        hei_table["HEI"] < 10,
        (hei_table["HEI"] >= 10) & (hei_table["HEI"] <= 20),
        hei_table["HEI"] > 20
    ]

    classes = [
        "Low Pollution",
        "Medium Pollution",
        "High Pollution"
    ]

    hei_table["HEI_Class"] = np.select(conditions, classes)

    return hei_table
def interpret_hei(hei_table):

    interpretation = {
        "Low Pollution": "Heavy metal contamination is minimal",
        "Medium Pollution": "Moderate heavy metal contamination",
        "High Pollution": "Severe heavy metal contamination"
    }

    hei_table["HEI_Interpretation"] = hei_table["HEI_Class"].map(interpretation)

    return hei_table


In [65]:
hei_calc = calculate_hei(df, limits)
hei_class = classify_hei(hei_calc)
hei_result = interpret_hei(hei_class)

print("\nHEI Calculated Values")
print(hei_calc)

print("\nHEI Classification")
print(hei_class[["Sample","HEI_Class"]])

print("\nHEI Interpretation")
print(hei_result[["Sample","HEI_Interpretation"]])


HEI Calculated Values
    Sample         HEI         HEI_Class                    HEI_Interpretation
0        0    3.138333     Low Pollution  Heavy metal contamination is minimal
1        1    3.486667     Low Pollution  Heavy metal contamination is minimal
2        2    3.821667     Low Pollution  Heavy metal contamination is minimal
3        3    2.833333     Low Pollution  Heavy metal contamination is minimal
4        4    2.508333     Low Pollution  Heavy metal contamination is minimal
5        5    7.951667     Low Pollution  Heavy metal contamination is minimal
6        6    6.391667     Low Pollution  Heavy metal contamination is minimal
7        7    4.721667     Low Pollution  Heavy metal contamination is minimal
8        8    3.343333     Low Pollution  Heavy metal contamination is minimal
9        9    2.280000     Low Pollution  Heavy metal contamination is minimal
10      10   16.830000  Medium Pollution    Moderate heavy metal contamination
11      11    1.478333     Lo

In [132]:
import pandas as pd

# Calculate HEI
hei_calc = calculate_hei(df, limits)

# Classify HEI values
hei_class = classify_hei(hei_calc)

# Interpret HEI classification
hei_result = interpret_hei(hei_class)

# Display results (as in your original code)
print("\nHEI Calculated Values")
print(hei_calc)

print("\nHEI Classification")
print(hei_class[["Sample","HEI_Class"]])

print("\nHEI Interpretation")
print(hei_result[["Sample","HEI_Interpretation"]])

# === EXPORT TO CSV ===

# Method 1: If all functions return DataFrames with same structure as sodium
hei_export = pd.DataFrame()

# Check which DataFrame has all the columns we need
# Usually the final result (hei_result) contains everything
hei_export["Sample"] = hei_result["Sample"]
hei_export["HEI_Value"] = hei_result["HEI"]  # Adjust column name as needed
hei_export["HEI_Class"] = hei_result["HEI_Class"]
hei_export["HEI_Interpretation"] = hei_result["HEI_Interpretation"]

# Export to CSV
hei_export.to_csv("HEI_Results.csv", index=False)

print("\n✅ HEI Results exported successfully to 'HEI_Results.csv'")
print(f"   Total samples: {len(hei_export)}")


HEI Calculated Values
    Sample         HEI         HEI_Class                    HEI_Interpretation
0        0    3.138333     Low Pollution  Heavy metal contamination is minimal
1        1    3.486667     Low Pollution  Heavy metal contamination is minimal
2        2    3.821667     Low Pollution  Heavy metal contamination is minimal
3        3    2.833333     Low Pollution  Heavy metal contamination is minimal
4        4    2.508333     Low Pollution  Heavy metal contamination is minimal
5        5    7.951667     Low Pollution  Heavy metal contamination is minimal
6        6    6.391667     Low Pollution  Heavy metal contamination is minimal
7        7    4.721667     Low Pollution  Heavy metal contamination is minimal
8        8    3.343333     Low Pollution  Heavy metal contamination is minimal
9        9    2.280000     Low Pollution  Heavy metal contamination is minimal
10      10   16.830000  Medium Pollution    Moderate heavy metal contamination
11      11    1.478333     Lo

In [68]:
def calculate_ci(df, limits):

    ci = pd.Series(0, index=df.index)

    for metal in limits:
        ci += df[metal] / limits[metal]

    ci_table = pd.DataFrame({
        "Sample": df.index,
        "CI": ci
    })

    return ci_table

def classify_ci(ci_table):

    conditions = [
        ci_table["CI"] < 1,
        (ci_table["CI"] >= 1) & (ci_table["CI"] <= 3),
        ci_table["CI"] > 3
    ]

    classes = [
        "Low",
        "Moderate",
        "High"
    ]

    ci_table["CI_Class"] = np.select(conditions, classes)

    return ci_table

def interpret_ci(ci_table):

    interpretation = {
        "Low": "No significant contamination",
        "Moderate": "Moderate heavy metal contamination",
        "High": "Serious heavy metal contamination"
    }

    ci_table["CI_Interpretation"] = ci_table["CI_Class"].map(interpretation)

    return ci_table

In [69]:
ci_calc = calculate_ci(df, limits)
ci_class = classify_ci(ci_calc)
ci_result = interpret_ci(ci_class)
print("\nCI Calculated Values")
print(ci_calc)
print("\nCI Classification")
print(ci_class[["Sample","CI_Class"]])
print("\nCI Interpretation")
print(ci_result[["Sample","CI_Interpretation"]])


CI Calculated Values
    Sample          CI  CI_Class                   CI_Interpretation
0        0    3.138333      High   Serious heavy metal contamination
1        1    3.486667      High   Serious heavy metal contamination
2        2    3.821667      High   Serious heavy metal contamination
3        3    2.833333  Moderate  Moderate heavy metal contamination
4        4    2.508333  Moderate  Moderate heavy metal contamination
5        5    7.951667      High   Serious heavy metal contamination
6        6    6.391667      High   Serious heavy metal contamination
7        7    4.721667      High   Serious heavy metal contamination
8        8    3.343333      High   Serious heavy metal contamination
9        9    2.280000  Moderate  Moderate heavy metal contamination
10      10   16.830000      High   Serious heavy metal contamination
11      11    1.478333  Moderate  Moderate heavy metal contamination
12      12    2.868333  Moderate  Moderate heavy metal contamination
13      13  

In [133]:
import pandas as pd

# Calculate CI
ci_calc = calculate_ci(df, limits)

# Classify CI values
ci_class = classify_ci(ci_calc)

# Interpret CI classification
ci_result = interpret_ci(ci_class)

# Display results (as in your original code)
print("\nCI Calculated Values")
print(ci_calc)

print("\nCI Classification")
print(ci_class[["Sample","CI_Class"]])

print("\nCI Interpretation")
print(ci_result[["Sample","CI_Interpretation"]])

# === EXPORT TO CSV ===

# Method 1: If all functions return DataFrames with same structure as sodium
ci_export = pd.DataFrame()

# Check which DataFrame has all the columns we need
# Usually the final result (ci_result) contains everything
ci_export["Sample"] = ci_result["Sample"]
ci_export["CI_Value"] = ci_result["CI"]  # Adjust column name as needed
ci_export["CI_Class"] = ci_result["CI_Class"]
ci_export["CI_Interpretation"] = ci_result["CI_Interpretation"]

# Export to CSV
ci_export.to_csv("CI_Results.csv", index=False)

print("\n✅ CI Results exported successfully to 'CI_Results.csv'")
print(f"   Total samples: {len(ci_export)}")


CI Calculated Values
    Sample          CI  CI_Class                   CI_Interpretation
0        0    3.138333      High   Serious heavy metal contamination
1        1    3.486667      High   Serious heavy metal contamination
2        2    3.821667      High   Serious heavy metal contamination
3        3    2.833333  Moderate  Moderate heavy metal contamination
4        4    2.508333  Moderate  Moderate heavy metal contamination
5        5    7.951667      High   Serious heavy metal contamination
6        6    6.391667      High   Serious heavy metal contamination
7        7    4.721667      High   Serious heavy metal contamination
8        8    3.343333      High   Serious heavy metal contamination
9        9    2.280000  Moderate  Moderate heavy metal contamination
10      10   16.830000      High   Serious heavy metal contamination
11      11    1.478333  Moderate  Moderate heavy metal contamination
12      12    2.868333  Moderate  Moderate heavy metal contamination
13      13  

In [ ]:
import pandas as pd

def build_index_table(
        df,
        index_values,
        classify_func,
        interpret_func,
        index_name,
        export=True,
        file_format="csv"
    ):
    
    """
    Generalized function for water quality indices

    Parameters
    ----------
    df : DataFrame
        Original dataset
    
    index_values : Series
        Calculated index values
    
    classify_func : function
        Function that classifies index
    
    interpret_func : function
        Function that interprets index
    
    index_name : str
        Name of index (SAR, KR, MH etc)
    
    export : bool
        Whether to export table
    
    file_format : str
        csv or excel
    """

    table = pd.DataFrame()

    table["Sample_No"] = df.index + 1
    table[index_name] = index_values

    table[f"{index_name}_Class"] = classify_func(index_values)

    table[f"{index_name}_Interpretation"] = interpret_func(index_values)

    if export:

        if file_format == "csv":
            table.to_csv(f"{index_name}_results.csv", index=False)

        elif file_format == "excel":
            table.to_excel(f"{index_name}_results.xlsx", index=False)

    return table